In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import xarray as xr

import zipfile as zp          # used for unzipping ppi files
from pathlib import Path      # used to play with pathnames to save 
from datetime import datetime # used to manipulate time :)


import wradlib as wr          # used for having fun with radar data

from PIL import Image         # used for creating gif loops
import os                     # used for retrieving file names

import h5py                   # used for reading .h5 files (Radar Level 1 data)
import h5netcdf               # used for converting .h5 files to NetCDF

# for labelling minutes of the day on a plot
import matplotlib.ticker as mticker

# # SPECIAL METHOD TO IMPORT LEROI RADAR GRIDDING PACKAGE AND CUSTOM FUNCTIONS FROM LOCAL DIRECTORY
import sys
sys.path.append('/home/563/sg3241/Notebooks/CustomFunctions')
from CustomFunctions1 import *

In [ ]:
# IMPORTED FROM EXAMPLE RADAR PYTHON SCRIPT

# import os #used for system commands
# import tempfile #used to create temporary folders to store data
# import zipfile #used to extract tar files
# import urllib #used to download data via http
# from datetime import datetime #used to manipulate time :)
# from glob import glob #used for manipulating pathnames

# import numpy as np 
# from matplotlib import pyplot as plt

# import pyart

# from matplotlib import animation
# from IPython.display import HTML
# import cartopy.crs as ccrs # A toolkit for map projections

In [ ]:
# FUNCTION
#pcolormesh but takes 1D X and Y coordinates for centres of the pixels

def pcolormeshC(x_centers, y_centers, z, ax=None,
                            shading='auto', **pcolor_kwargs):
    """
    Create a pcolormesh from a 2D array and 1D coordinate-center arrays.

    Parameters
    ----------
    x_centers : 1D array
        X coordinates of cell centers (length = number of columns in z)
    y_centers : 1D array
        Y coordinates of cell centers (length = number of rows in z)
    z : 2D array
        Data array with shape (len(y_centers), len(x_centers))
    ax : matplotlib.axes.Axes, optional
        Existing axis to draw on
    shading : str
        Passed to pcolormesh (default: 'auto')
    **pcolor_kwargs
        Extra kwargs passed to pcolormesh

    Returns
    -------
    pcm : QuadMesh
        The pcolormesh object
    """

    x_centers = np.asarray(x_centers)
    y_centers = np.asarray(y_centers)
    z = np.asarray(z)

    if z.shape != (len(y_centers), len(x_centers)):
        raise ValueError(
            f"z shape {z.shape} does not match "
            f"(len(y_centers), len(x_centers)) = "
            f"({len(y_centers)}, {len(x_centers)})"
        )

    # Convert centers -> edges
    def centers_to_edges(c):
        dc = np.diff(c)

        edges = np.empty(len(c) + 1)

        # Interior edges
        edges[1:-1] = c[:-1] + dc / 2

        # Extrapolate outer edges
        edges[0] = c[0] - dc[0] / 2
        edges[-1] = c[-1] + dc[-1] / 2

        return edges

    x_edges = centers_to_edges(x_centers)
    y_edges = centers_to_edges(y_centers)

    if ax is None:
        fig, ax = plt.subplots()

    pcm = ax.pcolormesh(
        x_edges,
        y_edges,
        z,
        shading=shading,
        **pcolor_kwargs
    )

    ax.set_xlabel("X")
    ax.set_ylabel("Y")

    return pcm

In [ ]:
# FUNCTION
# creates a histogram taking a 2D array and ignores nans AND ABSOLUTE VALUES ABOVE 10

def hist2d(arr, bins=50, density=False, ax=None, **hist_kwargs):
    """
    Create a histogram of all non-NaN values in a 2D NumPy array.

    Parameters
    ----------
    arr : np.ndarray
        Input 2D array.
    bins : int or sequence, optional
        Number of histogram bins or bin edges.
    density : bool, optional
        If True, normalize the histogram.
    ax : matplotlib.axes.Axes, optional
        Existing matplotlib axis to plot on.
    **hist_kwargs
        Extra keyword arguments passed to plt.hist().

    Returns
    -------
    hist : tuple
        Output from matplotlib hist():
        (counts, bin_edges, patches)
    """

    # Flatten array and remove NaNs 
    values = arr[~np.isnan(arr)]

    # Create axis if needed
    if ax is None:
        fig, ax = plt.subplots()

    # Plot histogram
    hist = ax.hist(values, bins=bins, density=density, **hist_kwargs)

    ax.set_xlabel("Value")
    ax.set_ylabel("Frequency" if not density else "Density")
    ax.set_title("Histogram of Non-NaN Values")

    return hist

In [ ]:
# FUNCTION
# creates a histogram taking a 2D array and ignores nans AND ABSOLUTE VALUES ABOVE 10

def hist2dsmall(arr, bins=50, density=False, ax=None, **hist_kwargs):
    """
    Create a histogram of all non-NaN values in a 2D NumPy array.

    Parameters
    ----------
    arr : np.ndarray
        Input 2D array.
    bins : int or sequence, optional
        Number of histogram bins or bin edges.
    density : bool, optional
        If True, normalize the histogram.
    ax : matplotlib.axes.Axes, optional
        Existing matplotlib axis to plot on.
    **hist_kwargs
        Extra keyword arguments passed to plt.hist().

    Returns
    -------
    hist : tuple
        Output from matplotlib hist():
        (counts, bin_edges, patches)
    """

    # Flatten array and remove NaNs OR ABSOLUTE VALUES ABOVE 10
    values = arr[~np.isnan(arr) & (np.abs(arr) <= 10)]

    # Create axis if needed
    if ax is None:
        fig, ax = plt.subplots()

    # Plot histogram
    hist = ax.hist(values, bins=bins, density=density, **hist_kwargs)

    ax.set_xlabel("Value")
    ax.set_ylabel("Frequency" if not density else "Density")
    ax.set_title("Histogram of Non-NaN Values")

    return hist

In [ ]:
# NOT A LOOP
# ALL IN ONE BLOCK

# the reference number for the radar location (ie 20 is Mackay)
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Bris)
GridOrPPI = 'ppi'

# the day in consideration (YYYYMMDD) and time (hhmmss)
# ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 3
RadarDay   = 9

RadarFileDate  = str(RadarYear).zfill(4) + str(RadarMonth).zfill(2) + str(RadarDay).zfill(2)
RadarFileTime =   '000000'



# THIS BLOCK TAKES A ZIPPED FOLDER FROM /G/DATA/ AND UNZIPS IT TO A FOLDER "nzippedRadarFiles" IN SCRATCH

# PATH NAMES
# ZippedFolder = '/g/data/rq0/level_1b/22/ppi/2024/'
ZippedFolder = '/g/data/rq0/level_1b/' + RadarIDno + '/' + GridOrPPI + '/' + str(RadarYear) + '/'
ZippedFile   =  RadarIDno + '_' + RadarFileDate + '_' + GridOrPPI + '.zip'

# place where the zipped file lives
ZippedPath = ZippedFolder + ZippedFile
# place to extract the files to
ExtractToDirectory = Path('/scratch/v46/sg3241/tmp/UnzippedRadarFiles/' + RadarIDno + '/' + RadarIDno + '_' + RadarFileDate + '_' + GridOrPPI + '/')

# Create the extraction folder (only if you haven't already)
if not Path(ExtractToDirectory).exists():
    ExtractToDirectory.mkdir(parents=True, exist_ok=True)

    # actually do the unzipping
    with zp.ZipFile(ZippedPath, 'r') as ZipReference:
        ZipReference.extractall(ExtractToDirectory)

    print('Done')
else:
    print('Unzipped Folder Already Exists')



# READ IN THE DATA TO AN XARRAY DATAFRAME

# the path of where the UNZIPPED radar data now live
RadarFolder = '/scratch/v46/sg3241/tmp/UnzippedRadarFiles/' + RadarIDno + '/' + RadarIDno + '_' + RadarFileDate + '_' + GridOrPPI + '/'
RadarFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + GridOrPPI + '.nc'

# open the radar data into an xarray data structure
RadarDataStruct = xr.open_dataset(RadarFolder + RadarFile, engine='netcdf4')


# Variables to plot in the 2D pcolormesh
Phis   = RadarDataStruct.elevation[::360] # retrieve set of all 13 elevation angles
Thetas = np.round(RadarDataStruct.azimuth[0:360])   # retrieve set of all 360 azimuth angles
Ranges = RadarDataStruct.range

# create a 13 elevation by 360 azimuth by 1283 range 3D array for ZDR
ZDR3D = np.reshape(np.array(RadarDataStruct.corrected_differential_reflectivity),[13,360,1283])

fig, ax = plt.subplots(figsize=(8,6))
RangeViewer = pcolormeshC(Thetas+180, Ranges*0.001, np.transpose(ZDR3D[1]), ax=ax, cmap='RdBu', vmin=-15, vmax=15)

ax.set_xlabel('Azimuth Angle [deg from North]')
ax.set_ylabel('Range [km]')
plt.colorbar(RangeViewer, ax=ax, label = 'Differential Reflectivity')
plt.grid()

plt.show()


hist2d(np.array(RadarDataStruct.corrected_differential_reflectivity))

In [ ]:
# THIS BLOCK IS WHERE THE USER PUTS INFO ABOUT THE RADAR

# the reference number for the radar location (ie 20 is Mackay)
RadarIDno = '106'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Bris)
GridOrPPI = 'ppi'

# the day in consideration (YYYYMMDD) and time (hhmmss)
# ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 11
RadarDay   = 9

RadarFileDate  = str(RadarYear).zfill(4) + str(RadarMonth).zfill(2) + str(RadarDay).zfill(2)
RadarFileTime =   '000000'

In [ ]:
# THIS IS THE "ADMIN" TEST VERSION OF THE PPI FILE FOR 29 JANUARY 2024
# THIS IS THE "ADMIN" TEST VERSION OF THE PPI FILE FOR 29 JANUARY 2024
# THIS IS THE "ADMIN" TEST VERSION OF THE PPI FILE FOR 29 JANUARY 2024
# THIS IS THE "ADMIN" TEST VERSION OF THE PPI FILE FOR 29 JANUARY 2024
# THIS IS THE "ADMIN" TEST VERSION OF THE PPI FILE FOR 29 JANUARY 2024

# THIS BLOCK TAKES A ZIPPED FOLDER FROM /G/DATA/ AND UNZIPS IT TO A FOLDER "nzippedRadarFiles" IN SCRATCH


# THIS BLOCK IS WHERE THE USER PUTS INFO ABOUT THE RADAR

# the reference number for the radar location (ie 20 is Mackay)
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Bris)
GridOrPPI = 'ppi'
# the day in consideration (YYYYMMDD) and time (hhmmss)
# ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 1
RadarDay   = 29
RadarFileDate  = str(RadarYear).zfill(4) + str(RadarMonth).zfill(2) + str(RadarDay).zfill(2)
# RadarFileTime =   '000000'

# PATH NAMES
# ZippedFolder = '/g/data/rq0/level_1b/22/ppi/2024/'
ZippedFolder = '/g/data/rq0/level_1b/' + RadarIDno + '/' + GridOrPPI + '/' + str(RadarYear) + '/'
ZippedFile   =  RadarIDno + '_' + RadarFileDate + '_' + GridOrPPI + '.zip'

# place where the zipped file lives
# ZippedPath = ZippedFolder + ZippedFile

ZippedPath = '/g/data/rq0/admin/incoming/level_1b/22/ppi/2024/22_20240129_ppi.zip'

# place to extract the files to
ExtractToDirectory = Path('/scratch/v46/sg3241/tmp/UnzippedRadarFilesVadmin/' + RadarIDno + '/' + RadarIDno + '_' + RadarFileDate + '_' + GridOrPPI + 'Vadmin/')

# Create the extraction folder (only if you haven't already)
if not Path(ExtractToDirectory).exists():
    ExtractToDirectory.mkdir(parents=True, exist_ok=True)

    # actually do the unzipping
    with zp.ZipFile(ZippedPath, 'r') as ZipReference:
        ZipReference.extractall(ExtractToDirectory)

    print('Done')
else:
    print('Unzipped Folder Already Exists')

In [ ]:
# THIS IS THE "ADMIN" TEST VERSION OF THE PPI FILE FOR 29 JANUARY 2024
# THIS IS THE "ADMIN" TEST VERSION OF THE PPI FILE FOR 29 JANUARY 2024
# THIS IS THE "ADMIN" TEST VERSION OF THE PPI FILE FOR 29 JANUARY 2024
# THIS IS THE "ADMIN" TEST VERSION OF THE PPI FILE FOR 29 JANUARY 2024
# THIS IS THE "ADMIN" TEST VERSION OF THE PPI FILE FOR 29 JANUARY 2024

# READ IN THE DATA TO AN XARRAY DATAFRAME

RadarFileTime =   '000000'

# the path of where the UNZIPPED radar data now live
RadarFolder = '/scratch/v46/sg3241/tmp/UnzippedRadarFilesVadmin/' + RadarIDno + '/' + RadarIDno + '_' + RadarFileDate + '_' + GridOrPPI + 'Vadmin/'
RadarFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + GridOrPPI + '.nc'
RadarPath = RadarFolder + RadarFile



# open the radar data into an xarray data structure
RadarDataStruct = xr.open_dataset(RadarPath, engine='netcdf4')

In [ ]:
# READ IN THE DATA TO AN XARRAY DATAFRAME

# the path of where the UNZIPPED radar data now live
RadarFolder = '/scratch/v46/sg3241/tmp/UnzippedRadarFiles/' + RadarIDno + '/' + RadarIDno + '_' + RadarFileDate + '_' + GridOrPPI + '/'
RadarFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + GridOrPPI + '.nc'

# open the radar data into an xarray data structure
RadarDataStruct = xr.open_dataset(RadarFolder + RadarFile, engine='netcdf4')

In [ ]:
Phis   = RadarDataStruct.elevation[::360] # retrieve set of all 13 elevation angles
Thetas = np.round(RadarDataStruct.azimuth[0:360])   # retrieve set of all 360 azimuth angles
Ranges = RadarDataStruct.range

# create a 13 elevation by 360 azimuth by 1283 range 3D array for ZDR
ZDR3D = np.reshape(np.array(RadarDataStruct.corrected_differential_reflectivity),[13,360,1283])

fig, ax = plt.subplots(figsize=(18,6))
RangeViewer = pcolormeshC(np.arange(0,360,1), Ranges*0.001, np.transpose(ZDR3D[6]), ax=ax, cmap='RdBu', vmin=-3, vmax=3)

ax.set_xlabel('Azimuth Angle [deg from North]')
ax.set_ylabel('Range [km]')
plt.colorbar(RangeViewer, ax=ax, label = 'Differential Reflectivity')
plt.grid()

plt.ylim([0,60])

plt.show()

In [ ]:
# Create a plot of the reflectivity for each 5-min period

fig, ax = plt.subplots(figsize=(8,6))
GridViewer = pcolormeshC(RadarDataStruct.x*0.001, RadarDataStruct.y*0.001, RadarDataStruct.corrected_reflectivity[0,2,:,:], ax=ax, cmap='nipy_spectral', vmin=-10, vmax=50)
# mutiply by 0.001 to get distances in km

ax.set_xlabel('East-West Distance [km]')
ax.set_ylabel('North-South Distance [km]')
plt.title('Radar Site ' + str(RadarIDno) + ' Reflectivity on ' + RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] + ' at ' + \
          str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6])
plt.colorbar(GridViewer, ax=ax, label = 'Reflectivity [dBZ]')
plt.grid()

SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/' + RadarIDno + '/' + RadarIDno + '_' + RadarFileDate + '_' + GridOrPPI + '/'
SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + GridOrPPI + '500m.png'

SavePath = SaveFolder + SaveFile

if not Path(SaveFolder).exists():
    print('doing')
    Path(SaveFolder).mkdir(parents=True, exist_ok=True)

# plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)

In [ ]:
# ALL IN ONE BLOCK TO LOOP

# THIS BLOCK IS WHERE THE USER PUTS INFO ABOUT THE RADAR

# the reference number for the radar location (ie 20 is Mackay)
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Bris)
GridOrPPI = 'grid'

if (RadarIDno == 22):
    RadarName = 'Mackay'
elif (RadarIDno == 106):
    RadarName = 'Townsville'
elif (RadarIDno == 2):
    RadarName = 'Melbourne'
elif (RadarIDno == 19):
    RadarName = 'Cairns'
elif (RadarIDno == 50):
    RadarName = 'Marburg'
elif (RadarIDno == 66):
    RadarName = 'Mount Staplyton'
else:
    RadarName = str(RadarIDno)

# the day in consideration (YYYYMMDD) and time (hhmmss)
# ALL IN UTC !!!
RadarYear  = 2024
# RadarMonth = 11
# RadarDay   = 7

MonthStart = [1, 1, 9, 1, 1, 1, 1, 1, 1, 1, 1, 1] # which day of the month do you want to start reading data from?
DaysInMonth = [31, 29, 9, 30, 31, 30, 31, 31, 30, 31, 30, 31] # number of days in each month of the year

for RadarMonth in range(3,4):
    for RadarDay in range(MonthStart[RadarMonth-1],DaysInMonth[RadarMonth-1] + 1):

        RadarFileDate  = str(RadarYear).zfill(4) + str(RadarMonth).zfill(2) + str(RadarDay).zfill(2)  # UTC date of the radar data
        RadarFileDatePrint = RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] # add a string of format YYYY-MM-DD for printing

        # THIS BLOCK TAKES A ZIPPED FOLDER FROM /G/DATA/ AND UNZIPS IT TO A FOLDER "nzippedRadarFiles" IN SCRATCH
                
        # PATH NAMES
        # ZippedFolder = '/g/data/rq0/level_1b/22/ppi/2024/'
        ZippedFolder = '/g/data/rq0/level_1b/' + RadarIDno + '/' + GridOrPPI + '/' + str(RadarYear) + '/'
        ZippedFile   =  RadarIDno + '_' + RadarFileDate + '_' + GridOrPPI + '.zip'
        
        # place where the zipped file lives
        ZippedPath = ZippedFolder + ZippedFile
        # place to extract the files to
        ExtractToDirectory = Path('/scratch/v46/sg3241/tmp/UnzippedRadarFiles/' + RadarIDno + '/' + RadarIDno + '_' + RadarFileDate + '_' + GridOrPPI + '/')
        
        # Create the extraction folder (only if you haven't already)
        if not Path(ExtractToDirectory).exists():
            ExtractToDirectory.mkdir(parents=True, exist_ok=True)

            with zp.ZipFile(ZippedPath, 'r') as ZipReference:
                ZipReference.extractall(ExtractToDirectory)
            print('Unzipped into Scratch')
        else:
            print('Unzipped Folder Already Exists')
    
        # actually do the unzipping
        if Path(ZippedPath).is_file():

            # loop over every 5 min period in the day
            for houri in range(0,24):
                for mini in range(0,60,5):
                    RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00' # write out the time in 6 digits (like 012040 for 01:20:40 AM)
                    RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] # add a string of format hh:mm:ss for printing
                    # print(RadarFileTime)
                
                    # READ IN THE DATA TO AN XARRAY DATAFRAME
                    
                    # the path of where the UNZIPPED radar data now live
                    RadarFolder = '/scratch/v46/sg3241/tmp/UnzippedRadarFiles/' + RadarIDno + '/' + RadarIDno + '_' + RadarFileDate + '_' + GridOrPPI + '/'
                    RadarFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + GridOrPPI + '.nc'
                    
                    # open the radar data into an xarray data structure
            
                    file_path = Path(RadarFolder + RadarFile)
                    if not file_path.is_file():
                        print('No file found for ' + RadarFolder + RadarFile)
                    else:
            
                        RadarDataStruct = xr.open_dataset(RadarFolder + RadarFile, engine='netcdf4')
                        
                        
                        
                        
                        
                        # Create a plot of the reflectivity for each 5-min period
                        
                        fig, ax = plt.subplots(figsize=(8,6))
                        GridViewer = pcolormeshC(RadarDataStruct.x*0.001, RadarDataStruct.y*0.001, RadarDataStruct.corrected_reflectivity[0,2,:,:], ax=ax, cmap='nipy_spectral', vmin=-10, vmax=50)
                        # mutiply by 0.001 to get distances in km
                        
                        ax.set_xlabel('East-West Distance [km]')
                        ax.set_ylabel('North-South Distance [km]')
                        plt.title('Radar Site ' + str(RadarIDno) + ' Reflectivity on ' + RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] + ' at ' + \
                                  str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] + ' UTC')
                        plt.colorbar(GridViewer, ax=ax, label = 'Reflectivity [dBZ]')
                        plt.grid()
                        
                        SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/' + RadarIDno + '/' + RadarIDno + '_' + RadarFileDate + '_' + GridOrPPI + '/'
                        SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_' + GridOrPPI + '500m.png'
                        
                        SavePath = SaveFolder + SaveFile
                        
                        if not Path(SaveFolder).exists():
                            Path(SaveFolder).mkdir(parents=True, exist_ok=True)
                                
                        plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)
                        plt.close()
                        
                        print('saved ' + RadarFileTimePrint)
                
            # THIS BLOCK CREATES A GIF FROM THE SAVED RADAR REFLECTIVITY PNG IMAGES
                
            # LOADING IMAGES
            files = sorted(os.listdir(SaveFolder)) # takes all of the files in the folder in the order they are named
            images = [
                Image.open(os.path.join(SaveFolder, f))
                for f in files
                if f.lower().endswith((".png", ".jpg", ".jpeg"))
            ]
            
            GIFsaveFolder = '/scratch/v46/sg3241/tmp/gifImages/' + RadarIDno + '/' + GridOrPPI + '/'
            GIFsaveFile = RadarIDno + '_' + RadarFileDate + '_' + GridOrPPI + '500m.gif'
            
            GIFsavePath = GIFsaveFolder + GIFsaveFile
            
            if not Path(GIFsaveFolder).exists():
                Path(GIFsaveFolder).mkdir(parents=True, exist_ok=True)
            
            # Save as looping GIF
            images[0].save(
                GIFsavePath,
                save_all=True,
                append_images=images[1:],
                duration=200,    # ms per frame
                loop=0,          # 0 = loop forever
            )
            print('Saved GIF for ' + RadarFileDatePrint)


            
        else:
            print('FAILED TO FIND ZIP FILE: ' + ZippedPath)


In [ ]:
# THIS BLOCK CREATES A GIF FROM THE SAVED RADAR REFLECTIVITY PNG IMAGES

# LOADING IMAGES
files = sorted(os.listdir(SaveFolder)) # takes all of the files in the folder in the order they are named
images = [
    Image.open(os.path.join(SaveFolder, f))
    for f in files
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
]

GIFsaveFolder = '/scratch/v46/sg3241/tmp/gifImages/' + RadarIDno + '/' + GridOrPPI + '/'
GIFsaveFile = RadarIDno + '_' + RadarFileDate + '_' + GridOrPPI + '500m.gif'

GIFsavePath = GIFsaveFolder + GIFsaveFile

if not Path(GIFsaveFolder).exists():
    Path(GIFsaveFolder).mkdir(parents=True, exist_ok=True)

# Save as looping GIF
images[0].save(
    GIFsavePath,
    save_all=True,
    append_images=images[1:],
    duration=200,    # ms per frame
    loop=0,          # 0 = loop forever
)

In [ ]:
# turn the corrected differential reflectivity into a numpy array
ZDRarray = np.array(RadarDataStruct.corrected_differential_reflectivity)

# collect only those ZDR values that are not -15 or nan
# returns the values of those ZDRs and the indices for where they are in the original data
SigZDR, SigZDRinds = zip(*[(x,i) for i, x in enumerate(ZDRarray[0]) if x > -14])

In [ ]:
Phis   = RadarDataStruct.elevation[::360] # retrieve set of all 13 elevation angles
Thetas = np.round(RadarDataStruct.azimuth[0:360])   # retrieve set of all 360 azimuth angles
Ranges = RadarDataStruct.range

In [ ]:
# create a 13 elevation by 360 azimuth by 1283 range 3D array for ZDR
ZDR3D = np.reshape(np.array(RadarDataStruct.corrected_differential_reflectivity),[13,360,1283])

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
RangeViewer = pcolormeshC(Thetas+180, Ranges*0.001, np.transpose(ZDR3D[0]), ax=ax, cmap='RdBu', vmin=-3, vmax=3)

ax.set_xlabel('Azimuth Angle [deg from North]')
ax.set_ylabel('Range [km]')
plt.colorbar(RangeViewer, ax=ax, label = 'Differential Reflectivity')
plt.grid()

plt.show()

In [ ]:
# plot a quick view of the elevation angle over all "4680" times


elev = np.array(RadarDataStruct.elevation)
x = np.arange(0,len(elev),1)

plt.figure()
plt.plot(x, elev)
plt.grid()

plt.xlim([0,len(elev)])
plt.ylim([0,45])
plt.title('Scanning Elevation Angle for each 4680 times')

azi = np.array(RadarDataStruct.azimuth)
x2 = np.arange(0,len(azi),1)

plt.figure()
plt.plot(x2, azi)
plt.grid()

plt.xlim([0,len(azi)])
plt.ylim([-180,180])
plt.title('Scanning Azimuth (from N) Angle for each 4680 times')

In [ ]:
# LEVEL 2 DATA VIEWER

# Choose LEVEL 2 variable to view (both ALL CAPS and all lower case are used in the file names)
# (options are         'AZSHEAR', 'COLUMNMAXREFLECTIVITY',   'ECHOTOPHEIGHT', 'REFLECTIVITY',               'SHI', and                     'STEINER')
# (options are (Azimuthal Shear),          (Column Max Z), (Echo Top Height),    (Ground? Z), (Severe Hail Index), and 'Steiner Echo Classification')
VARIABLEupper = 'STEINER'
VARIABLElower = 'steiner'

# path to the radar data netcdf
NCFolder = '/g/data/rq0/level_2/' + RadarIDno + '/' + VARIABLEupper + '/' # + str(RadarYear) + '/'
NCFile   =  RadarIDno + '_' + RadarFileDate + '_' + VARIABLElower + '.nc'

# place where the netcdf file lives
NCPath = NCFolder + NCFile

RadarDataStruct = xr.open_dataset(NCFolder + NCFile, engine='netcdf4')

RadarDataStruct

In [ ]:
# THIS BLOCK IS WHERE THE USER PUTS INFO ABOUT THE RADAR

# the reference number for the radar location (ie 20 is Mackay)
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Bris)
GridOrPPI = 'grid'

# the day in consideration (YYYYMMDD) and time (hhmmss)
# ALL IN UTC !!!
RadarYear  = 2026
RadarMonth = 4
RadarDay   = 30

RadarFileDate  = str(RadarYear).zfill(4) + str(RadarMonth).zfill(2) + str(RadarDay).zfill(2)
RadarFileTime =   '120000'

In [ ]:
# LEVEL 1 DATA unzipper

# path to the radar data zip files
ZipFolder = '/g/data/rq0/level_1/odim_pvol/' + RadarIDno + '/' + str(RadarYear) + '/vol/'
ZipFile   =  RadarIDno + '_' + RadarFileDate + '.pvol.zip'

# place where the Zip file lives
ZipPath = ZipFolder + ZipFile

# place to extract the files to
ExtractToDirectory = Path('/scratch/v46/sg3241/tmp/UnzippedRadarFiles/Level1/vol/'  + RadarIDno + '/' + RadarIDno + '_' + RadarFileDate + '_pvol/')

# Create the extraction folder (only if you haven't already)
if not Path(ExtractToDirectory).exists():
    ExtractToDirectory.mkdir(parents=True, exist_ok=True)

    # actually do the unzipping
    with zp.ZipFile(ZipPath, 'r') as ZipReference:
        ZipReference.extractall(ExtractToDirectory)

    print('Done')
else:
    print('Unzipped Folder Already Exists')

In [ ]:
# LEVEL 1 DATA viewer

# the path of where the UNZIPPED radar data now live
RadarFolder = '/scratch/v46/sg3241/tmp/UnzippedRadarFiles/Level1/vol/' + RadarIDno + '/' + RadarIDno + '_' + RadarFileDate + '_pvol/'
RadarFile   = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '.pvol.h5'

RadarPath = RadarFolder + RadarFile

# scratch/v46/sg3241/tmp/UnzippedRadarFiles/Level1/vol/22/22_20240101_pvol/22_20240101_000500.pvol.h5

#Open the H5 file in read mode
with h5py.File(RadarPath, 'r') as f:

    def print_structure(name, obj):

        print(name, type(obj))

    f.visititems(print_structure)

In [ ]:
# INSPECT THE VALUES OF THE VARIABLE

with h5py.File(RadarPath, 'r') as f:

    d = f['dataset1/data1/data']

    print("Shape:", d.shape)

    print("Dtype:", d.dtype)

    data = d[:]

    print(data)

In [ ]:
# CHECK WHAT VARIABLE YOU ARE DEALING WITH

with h5py.File(RadarPath, 'r') as f:

    what = f['dataset1/data1/what']

    for k, v in what.attrs.items():

        print(k, v)

In [ ]:
# THIS BLOCK TAKES A ZIPPED FOLDER FROM /G/DATA/ AND UNZIPS IT TO A FOLDER "nzippedRadarFiles" IN SCRATCH

# PATH NAMES
# ZippedFolder = '/g/data/rq0/level_1b/22/ppi/2024/'
ZippedFolder = '/g/data/rq0/level_1b/' + RadarIDno + '/' + GridOrPPI + '/' + str(RadarYear) + '/'
ZippedFile   =  RadarIDno + '_' + RadarFileDate + '_' + GridOrPPI + '.zip'

# place where the zipped file lives
ZippedPath = ZippedFolder + ZippedFile
# place to extract the files to
ExtractToDirectory = Path('/scratch/v46/sg3241/tmp/UnzippedRadarFiles/' + RadarIDno + '/' + RadarIDno + '_' + RadarFileDate + '_' + GridOrPPI + '/')

# Create the extraction folder (only if you haven't already)
if not Path(ExtractToDirectory).exists():
    ExtractToDirectory.mkdir(parents=True, exist_ok=True)

    # actually do the unzipping
    with zp.ZipFile(ZippedPath, 'r') as ZipReference:
        ZipReference.extractall(ExtractToDirectory)

    print('Done')
else:
    print('Unzipped Folder Already Exists')

In [ ]:
# CONVERTS STORED VALUES TO REAL VALUES

with h5py.File(RadarPath, 'r') as f:

    raw = f['dataset7/data1/data'][:]
    attrs = f['dataset7/data1/what'].attrs
    gain = attrs['gain']
    offset = attrs['offset']

    nodata = attrs['nodata']

    data = raw.astype(float)
    data[raw == nodata] = np.nan
    data = offset + gain * data

    print(data)

In [ ]:
with h5netcdf.File("mydata.nc", "w") as f:
    # set dimensions with a dictionary
    f.dimensions = {"x": 5}
    # and update them with a dict-like interface
    # f.dimensions['x'] = 5
    # f.dimensions.update({'x': 5})

    v = f.create_variable("hello", ("x",), float)
    v[:] = np.ones(5)

    # you don't need to create groups first
    # you also don't need to create dimensions first if you supply data
    # with the new variable
    v = f.create_variable("/grouped/data", ("y",), data=np.arange(10))

    # access and modify attributes with a dict-like interface
    v.attrs["foo"] = "bar"

    # you can access variables and groups directly using a hierarchical
    # keys like h5py
    print(f["/grouped/data"])

    # add an unlimited dimension
    f.dimensions["z"] = None
    # explicitly resize a dimension and all variables using it
    f.resize_dimension("z", 3)

In [ ]:
# HOVMOLLER ATTEMPT

import warnings
warnings.filterwarnings("ignore")
print('Warning: ignoring warnings!')

# "/scratch/v46/sg3241/tmp/UnzippedRadarFiles/22/22_20240309_ppi/22_20240309_001500_ppi.nc"

# CHOOSE THE RADAR
RadarIDno = '22'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Bris)

# CHOOSE THE DATE
# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 2
RadarDay   = 25

# choose the radar azimuth where you are looking
AziAngle = 290.0 # degrees clockwise from North

# choose an elevation angle to look at
EleAngle = 1.4
# options are 0.5,  0.8,  1.4,  2.4,  3.5,  4.7,  6.0 ,  7.8, 10.0 , 13.0 , 17.0 , 23.0 , 32.0

# PLEASE MAKE IT SO THE PLOT VARIABLES CAN BE CHOSEN HERE!!!! (ADD STRING EXECUTERS AND SUCH)
# CHOOSE THE VARIABLE TO PLOT
Var  = 'Z'

# LIST OF POSSIBLE VARIABLES
# [Z]         'corrected_reflectivity'
# [CC]        'corrected_cross_correlation_ratio'
# [ZDR]       'corrected_differential_reflectivity'
# [KDP]       'corrected_specific_differential_phase'
# [PhiDP]     'corrected_differential_phase'
# [IntAtt]    'path_integrated_attenuation'
# [DifIntAtt] 'path_integrated_differential_attenuation'
# [EchClas]   'radar_echo_classification'
# [V]         'corrected_velocity'
# [AzSh]      'azshear'

# USER CHOICE FOLLOW-ON SECTION

# radar choice follow-on
if (RadarIDno == '22'):
    RadarSiteName = 'Mackay'
elif (RadarIDno == '106'):
    RadarSiteName = 'Townsville'
elif (RadarIDno == '66'):
    RadarSiteName = 'Mt Staplyton'
elif (RadarIDno == '50'):
    RadarSiteName = 'Marabong'
else:
    RadarSiteName = 'Site ' + RadarIDno


# date choice follow-on
# add leading zeros for strings
YYYY = str(RadarYear).zfill(4)
MM = str(RadarMonth).zfill(2)
DD = str(RadarDay).zfill(2)
# write out the data in one string with and without dashes
RadarFileDate  = YYYY + MM + DD
RadarFileDatePrint = YYYY + '-' + MM + '-' + DD

# variable choice follow-on
if (Var == 'Z'):
    VarName     = 'Reflectivity'
    VarNameLong = 'corrected_reflectivity'
    VarMinVal = -10 # [dBZ]
    VarMaxVal =  60 # [dBZ]
    VarUnit   = 'dBZ'
    VarColourBar = 'nipy_spectral'
elif (Var == 'ZDR'):
    VarName     = 'Differential Reflectivity'
    VarNameLong = 'corrected_differential_reflectivity'
    VarMinVal = -5 # [dB]
    VarMaxVal =  5 # [dB]
    VarUnit   = 'dB'
    VarColourBar = 'RdBu'
elif (Var == 'CC'):
    VarName     = 'Correlation Coefficient'
    VarNameLong = 'corrected_cross_correlation_ratio'
    VarMinVal = 0.8 # [0 to 1]
    VarMaxVal = 1.0 # [0 to 1]
    VarUnit   = '-0 to 1'
    VarColourBar = 'nipy_spectral'
elif (Var == 'KDP'):
    VarName     = 'Specific Differential Phase'
    VarNameLong = 'corrected_specific_differential_phase'
    VarMinVal = 0  # [deg/ km]
    VarMaxVal = 10 # [deg / km]
    VarUnit   = 'deg / km'
    VarColourBar = 'nipy_spectral'
elif (Var == 'PhiDP'):
    VarName     = 'Differential Phase'
    VarNameLong = 'corrected_differential_phase'
    VarMinVal = 0  # [deg]
    VarMaxVal = 30 # [deg]
    VarUnit   = 'deg'
    VarColourBar = 'nipy_spectral'
elif (Var == 'V'):
    VarName     = 'Velocity'
    VarNameLong = 'corrected_velocity'
    VarMinVal = -30 # [m/s]
    VarMaxVal =  30 # [m/s]
    VarUnit   = 'm/s'
    VarColourBar = 'RdBu_r'
else:
    raise ValueError("Input Variable '" + Var + "' not available\n" + "Please choose from the following list:\n" + \
          "[Z] 'corrected_reflectivity', [CC] 'corrected_cross_correlation_ratio', [ZDR] 'corrected_differential_reflectivity'\n" + \
          "[KDP] 'corrected_specific_differential_phase', [PhiDP] 'corrected_differential_phase'")

NumTimes = int(1440 / 5) # number of times in a day (288 sets of 5 min here)
NumRanges = 1283    # number of ranges in the net cdf ppi files

# create an empty array to store what will be plotted in the hovmoller diagram
HovGrid      = np.full([NumTimes,NumRanges], np.nan)
# create an empty array to store the times associated with each data point in the hovmoller diagram
HovGridTimes =  times = np.full(NumTimes, np.datetime64('NaT'), dtype='datetime64[ns]')

# index of 5 min intervals
mini5 = 0
# LOOP OVER EVERY 5 MIN PERIOD IN THE DAY
for houri in range(0,24):
    for mini in range(0,60,5):
        
        RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00' # write out the time in 6 digits (like 012040 for 01:20:40 AM)
        RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] 
        
        # add a string of format hh:mm:ss for printing
        print('working on ' + RadarFileTimePrint)
    
        NetCDFstoragePath = '/scratch/v46/sg3241/tmp/UnzippedRadarFiles/' + RadarIDno + '/' + \
                                RadarIDno + '_' + RadarFileDate + '_ppi/' + \
                                RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_ppi.nc'
    
        # try to load in the netcdf file and if it doesn't work, just keep going through the loop
        try:
            xgrid = xr.open_dataset(NetCDFstoragePath)
        except FileNotFoundError:
            print(f'File missing for {RadarFileTimePrint}, skipping: {NetCDFstoragePath}')
            continue
    
        
        HovGrid[mini5, :] = xgrid[VarNameLong][ np.where( (xgrid.azimuth == AziAngle) & (xgrid.elevation == EleAngle) ) ]
        
        HovGridTimes[mini5] = np.array(xgrid.time[ np.where( (xgrid.azimuth == AziAngle) & (xgrid.elevation == EleAngle) ) ])[0]
        # print(np.array(xgrid.time[ np.where( (xgrid.azimuth == AziAngle) & (xgrid.elevation == EleAngle) ) ])[0])
    
        mini5 = mini5 + 1
        

In [ ]:
xgrid.range.values

In [ ]:
# PLOT THE HOVMOLLER DIAGRAM

# Convert times to minutes since midnight of that day

PlotVar = 'corrected_reflectivity'

# Build start-of-day as numpy datetime64 (UTC)
start_of_day = np.datetime64(RadarFileDatePrint + 'T00:00')
HovGridMins = (HovGridTimes - start_of_day).astype('timedelta64[m]').astype(float)

fig, ax = plt.subplots(figsize=(8,6))
HovViewer = pcolormeshC(xgrid.range*0.001, HovGridMins, HovGrid, ax=ax, cmap=VarColourBar, vmin=VarMinVal, vmax=VarMaxVal)

# Reverse time axis: 0 at top, 24 h at bottom
ax.invert_yaxis()

# Reverse time axis: radar in the east, looking west
ax.invert_xaxis()

ax.set_xlabel('Range [km]')
ax.set_ylabel('Time (UTC)')

plt.title(VarName + ' Hovmoller Diagram\nfor ' + RadarSiteName + ' Radar on ' + \
          RadarFileDate[0:4] + '-' + RadarFileDate[4:6] + '-' + RadarFileDate[6:8] + \
          '\nfor the ' + str(EleAngle) + ' Degree Elevation Angle at ' + str(AziAngle) + ' Degrees Azimuth')

# Tick every hour (60 minutes)
ax.yaxis.set_major_locator(mticker.MultipleLocator(120))

plt.xlim([150,0])

# Format minutes as HH:MM
def minutes_to_hhmm(m, pos):
    m = int(m)
    h = m // 60
    mm = m % 60
    return f'{h:02d}:{mm:02d}'

ax.yaxis.set_major_formatter(mticker.FuncFormatter(minutes_to_hhmm))

plt.grid()

plt.colorbar(HovViewer, ax=ax, label= VarName + '[' + VarUnit + ']')
plt.tight_layout()

SaveFolder = '/scratch/v46/sg3241/tmp/pngImages/hov/' + RadarIDno + '/' + RadarFileDate + '/'
SaveFile   = RadarIDno + '_' + RadarFileDate + '_' + PlotVar + '_hovZOOM.png'

SavePath = SaveFolder + SaveFile

if not Path(SaveFolder).exists():
    print('Creating Folder: ' + SaveFolder)
    Path(SaveFolder).mkdir(parents=True, exist_ok=True)

# plt.savefig(SavePath, bbox_inches='tight', facecolor='w', dpi = 300)
# plt.close()